# Grocery Inflation Tracker: Data Exploration
**Goal:** Measure how much a typical Canadian family's grocery basket has risen since 2019, and which foods drove it.
**Data:** Statistics Canada, Monthly average retail prices for selected products (Table 18-10-0245-01), pulled via the StatCan Web Data Service API.

In [8]:
import pandas as pd, requests, io, zipfile

def get_table(pid):
    url = f"https://www150.statcan.gc.ca/t1/wds/rest/getFullTableDownloadCSV/{pid}/en"
    zip_url = requests.get(url).json()["object"]
    z = zipfile.ZipFile(io.BytesIO(requests.get(zip_url).content))
    return pd.read_csv(z.open(f"{pid}.csv"))

prices = get_table("18100245")
prices.head()


C:\Users\Zia\AppData\Local\Temp\ipykernel_5152\2749523224.py:7: DtypeWarning: Columns (11: STATUS) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(z.open(f"{pid}.csv"))


,REF_DATE,GEO,DGUID,Products,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2017-01,Canada,2021A000011124,"Beef stewing cuts, per kilogram",Dollars,81,units,0,v1353834271,11.10,12.66,NaN,NaN,NaN,2
1,2017-01,Canada,2021A000011124,"Beef striploin cuts, per kilogram",Dollars,81,units,0,v1353834272,11.20,21.94,NaN,NaN,NaN,2
2,2017-01,Canada,2021A000011124,"Beef top sirloin cuts, per kilogram",Dollars,81,units,0,v1353834273,11.30,13.44,NaN,NaN,NaN,2
3,2017-01,Canada,2021A000011124,"Beef rib cuts, per kilogram",Dollars,81,units,0,v1353834311,11.41,20.17,NaN,NaN,NaN,2
4,2017-01,Canada,2021A000011124,"Ground beef, per kilogram",Dollars,81,units,0,v1353834274,11.40,9.12,NaN,NaN,NaN,2


In [3]:
print(prices.shape)
print(prices.columns)

(142740, 15)
Index(['REF_DATE', 'GEO', 'DGUID', 'Products', 'UOM', 'UOM_ID',
       'SCALAR_FACTOR', 'SCALAR_ID', 'VECTOR', 'COORDINATE', 'VALUE', 'STATUS',
       'SYMBOL', 'TERMINATED', 'DECIMALS'],
      dtype='str')


## First look
The raw table has [142740] rows and [15] columns. The key columns are `REF_DATE` (month), `GEO` (province), `Products` (item and size), and `VALUE` (average price in $).


In [4]:
prices.head(10)

,REF_DATE,GEO,DGUID,Products,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2017-01,Canada,2021A000011124,"Beef stewing cuts, per kilogram",Dollars,81,units,0,v1353834271,11.10,12.66,NaN,NaN,NaN,2
1,2017-01,Canada,2021A000011124,"Beef striploin cuts, per kilogram",Dollars,81,units,0,v1353834272,11.20,21.94,NaN,NaN,NaN,2
2,2017-01,Canada,2021A000011124,"Beef top sirloin cuts, per kilogram",Dollars,81,units,0,v1353834273,11.30,13.44,NaN,NaN,NaN,2
3,2017-01,Canada,2021A000011124,"Beef rib cuts, per kilogram",Dollars,81,units,0,v1353834311,11.41,20.17,NaN,NaN,NaN,2
4,2017-01,Canada,2021A000011124,"Ground beef, per kilogram",Dollars,81,units,0,v1353834274,11.40,9.12,NaN,NaN,NaN,2
5,2017-01,Canada,2021A000011124,"Pork loin cuts, per kilogram",Dollars,81,units,0,v1353834275,11.50,7.34,NaN,NaN,NaN,2
6,2017-01,Canada,2021A000011124,"Pork rib cuts, per kilogram",Dollars,81,units,0,v1353834276,11.60,7.37,NaN,NaN,NaN,2
7,2017-01,Canada,2021A000011124,"Pork shoulder cuts, per kilogram",Dollars,81,units,0,v1353834312,11.42,4.76,NaN,NaN,NaN,2
8,2017-01,Canada,2021A000011124,"Whole chicken, per kilogram",Dollars,81,units,0,v1353834277,11.70,5.15,NaN,NaN,NaN,2
9,2017-01,Canada,2021A000011124,"Chicken breasts, per kilogram",Dollars,81,units,0,v1353834278,11.80,11.38,NaN,NaN,NaN,2


In [5]:
prices["GEO"].unique()

<StringArray>
[                            'Canada',          'Newfoundland and Labrador',
               'Prince Edward Island',                        'Nova Scotia',
                      'New Brunswick',                             'Quebec',
                            'Ontario',                           'Manitoba',
                       'Saskatchewan',                            'Alberta',
                   'British Columbia',                  'Whitehorse, Yukon',
 'Yellowknife, Northwest Territories']
Length: 13, dtype: str

In [6]:
print(prices["Products"].nunique(), "products")
prices["Products"].unique()

110 products


<StringArray>
[    'Beef stewing cuts, per kilogram',   'Beef striploin cuts, per kilogram',
 'Beef top sirloin cuts, per kilogram',         'Beef rib cuts, per kilogram',
           'Ground beef, per kilogram',        'Pork loin cuts, per kilogram',
         'Pork rib cuts, per kilogram',    'Pork shoulder cuts, per kilogram',
         'Whole chicken, per kilogram',       'Chicken breasts, per kilogram',
 ...
              'Salsa, 418 millilitres',        'Pasta sauce, 650 millilitres',
     'Salad dressing, 475 millilitres',                  'Almonds, 200 grams',
                  'Peanuts, 450 grams',         'Sunflower seeds, 400 grams ',
                 'Deodorant, 85 grams',         'Toothpaste, 100 millilitres',
            'Shampoo, 400 millilitres',      'Laundry detergent, 4.43 litres']
Length: 110, dtype: str

In [7]:
print("From:", prices["REF_DATE"].min())
print("To:", prices["REF_DATE"].max())
print("Missing prices:", prices["VALUE"].isna().sum())

From: 2017-01
To: 2026-09
Missing prices: 37


## What's in the data
- **Geography:** Canada plus [10] provinces and [13] territories. [Note any that are missing, e.g. no territories.]
- **Products:** [110] items, each with a fixed size (e.g. "Eggs, 1 dozen").
- **Time range:** [January] to [September].
- **Missing prices:** [37] rows. These are dropped in cleaning.

## Cleaning
Kept only the columns needed, renamed them, converted dates, and dropped rows with no price.

In [9]:
df = prices[["REF_DATE", "GEO", "Products", "VALUE"]].copy()
df.columns = ["date", "province", "product", "price"]
df["date"] = pd.to_datetime(df["date"])
df = df.dropna(subset=["price"])
df.head()

,date,province,product,price
0,2017-01-01,Canada,"Beef stewing cuts, per kilogram",12.66
1,2017-01-01,Canada,"Beef striploin cuts, per kilogram",21.94
2,2017-01-01,Canada,"Beef top sirloin cuts, per kilogram",13.44
3,2017-01-01,Canada,"Beef rib cuts, per kilogram",20.17
4,2017-01-01,Canada,"Ground beef, per kilogram",9.12


In [10]:
assert (df["price"] > 0).all(), "Found zero or negative prices"
print(df.duplicated(["date", "province", "product"]).sum(), "duplicate rows")
print(len(df), "rows after cleaning")

0 duplicate rows
142703 rows after cleaning


## Quality checks
- All prices are positive
- [0] duplicate rows found
- [142703] rows remain after cleaning